# Gemma 4 E4B × Nemotron-PII — 2-Hour Fine-Tune

Fine-tunes **Gemma 4 E4B** (via Unsloth) on the **nvidia/Nemotron-PII** dataset to teach the model on-device PII/PHI detection and tagging.

After training the notebook saves:
- `lora_adapter/` — HuggingFace LoRA weights (push to Hub or keep local)
- `gemma4-pii-unsloth.Q4_K_M.gguf` — ready for **cactus-compute/cactus** edge deployment

**Runtime:** A100 recommended (fits in ~14 GB VRAM with 4-bit). T4 works with batch-size 1.

---
**Deployment path:**
```
export CACTUS_DETECTION_MODEL_PATH=/path/to/gemma4-pii-unsloth.Q4_K_M.gguf
cargo run --release --features cactus -p masker-cli -- --backend cactus
```

## 0 · Mount Drive (to persist outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/gemma4-pii'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Outputs → {OUTPUT_DIR}')

## 1 · Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps transformers==5.5.0
!pip install trl datasets

## 2 · Configuration

In [ ]:
import torch, time

MODEL_NAME     = 'unsloth/gemma-4-E4B-it'
DATASET_NAME   = 'nvidia/Nemotron-PII'
MAX_SEQ_LENGTH = 2048
LORA_RANK      = 16
MAX_HOURS      = 2.0          # wall-clock training budget
BATCH_SIZE     = 2            # reduce to 1 on T4
GRAD_ACCUM     = 4
LEARNING_RATE  = 2e-4

USE_BF16 = torch.cuda.is_bf16_supported()
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'bf16: {USE_BF16}')

## 3 · Load Gemma 4 E4B with Unsloth

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
    full_finetuning = False,
)
print('Model loaded.')

## 4 · Add LoRA adapters

In [ ]:
model = FastModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_RANK * 2,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()

## 5 · Load & format Nemotron-PII dataset

Task: **given raw text → produce text with `<PII type="...">...</PII>` tags.**  
50k train / 50k test rows covering 55+ PII/PHI categories across 50+ industries.

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    'You are a privacy intelligence expert. '
    'Identify and tag every PII/PHI entity in the input text. '
    'Wrap each entity with <PII type="entity_type">value</PII> tags. '
    'Common types: name, email, phone, ssn, address, dob, mrn, '
    'insurance_id, account_number, credit_card, passport, ip_address, '
    'url, organization, username, npi, license_number.'
)

raw_dataset = load_dataset(DATASET_NAME, split='train')
print(f'Raw dataset: {len(raw_dataset):,} rows')
print('Columns:', raw_dataset.column_names)
print('\nSample text snippet:')
print(raw_dataset[0]['text'][:300])
print('\nSample tagged snippet:')
print(raw_dataset[0]['text_tagged'][:400])

In [ ]:
def format_row(row):
    messages = [
        {'role': 'user',      'content': SYSTEM_PROMPT + '\n\n' + row['text']},
        {'role': 'assistant', 'content': row['text_tagged']},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = raw_dataset.map(
    format_row,
    remove_columns=raw_dataset.column_names,
    num_proc=4,
    desc='Formatting',
)
print(f'Formatted: {len(train_dataset):,} examples')
print('\nFirst formatted example (first 500 chars):')
print(train_dataset[0]['text'][:500])

## 6 · Train (2-hour wall-clock budget)

In [ ]:
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

class TimeLimitCallback(TrainerCallback):
    def __init__(self, max_seconds):
        self.max_seconds = max_seconds
        self._start = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._start = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self._start
        if elapsed >= self.max_seconds:
            print(f'\n[{elapsed/3600:.2f}h] Time limit reached — stopping.')
            control.should_training_stop = True
        return control


trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_dataset,
    args = SFTConfig(
        output_dir                  = OUTPUT_DIR,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        max_seq_length              = MAX_SEQ_LENGTH,
        num_train_epochs            = 10,      # stopped early by callback
        learning_rate               = LEARNING_RATE,
        warmup_steps                = 20,
        lr_scheduler_type           = 'cosine',
        bf16                        = USE_BF16,
        fp16                        = not USE_BF16,
        logging_steps               = 10,
        save_steps                  = 100,
        save_total_limit            = 3,
        dataset_text_field          = 'text',
        dataset_num_proc            = 4,
        report_to                   = 'none',
    ),
    callbacks = [TimeLimitCallback(MAX_HOURS * 3600)],
)

print(f'Starting {MAX_HOURS:.0f}-hour training run…')
trainer_stats = trainer.train()

## 7 · Save LoRA adapter

In [ ]:
lora_path = f'{OUTPUT_DIR}/lora_adapter'
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f'LoRA adapter saved → {lora_path}')

## 8 · Export GGUF for cactus-compute/cactus

Q4_K_M is the recommended quantisation for edge deployment — balances quality and size (~2.5 GB for E4B).

In [ ]:
gguf_prefix = f'{OUTPUT_DIR}/gemma4-pii'
print('Exporting Q4_K_M GGUF (this takes a few minutes)…')
model.save_pretrained_gguf(
    gguf_prefix,
    tokenizer,
    quantization_method = 'q4_k_m',
)
gguf_path = f'{gguf_prefix}-unsloth.Q4_K_M.gguf'
print(f'\nGGUF saved → {gguf_path}')

## 9 · Quick inference test

In [ ]:
from unsloth import FastModel
from transformers import TextStreamer

# Switch model to inference mode
FastModel.for_inference(model)

test_text = (
    'Hi, my name is Sarah Johnson and my SSN is 123-45-6789. '
    'Please call me at (415) 555-0147 or email sarah.j@example.com. '
    'My MRN is MRN-88291 and insurance ID BCBS-443221.'
)

messages = [
    {'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + test_text},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

print('Input:')
print(test_text)
print('\nModel output:')
_ = model.generate(
    input_ids       = inputs,
    max_new_tokens  = 512,
    do_sample       = False,
    streamer        = TextStreamer(tokenizer, skip_prompt=True),
)

## 10 · Download GGUF to local machine

The file is saved to Google Drive. You can also download it directly:

In [ ]:
from google.colab import files
import os

gguf_path = f'{OUTPUT_DIR}/gemma4-pii-unsloth.Q4_K_M.gguf'
size_gb = os.path.getsize(gguf_path) / 1e9
print(f'GGUF file: {size_gb:.2f} GB')
print('Downloading…')
files.download(gguf_path)

## 11 · Load on cactus-compute/cactus

Once you have the `.gguf` file locally, run:

```bash
# Copy into cactus weights dir
cp gemma4-pii-unsloth.Q4_K_M.gguf ~/.cactus/weights/gemma4-pii.gguf

# Point masker-core at the fine-tuned model
export CACTUS_DETECTION_MODEL_PATH=~/.cactus/weights/gemma4-pii.gguf
export CACTUS_STT_MODEL_PATH=~/.cactus/weights/whisper-small

# Run masker with on-device PII detection
cd platform/masker-core
cargo run --release --features cactus -p masker-cli -- --backend cactus

# Or run a single turn
masker run-turn \
  --text "My SSN is 123-45-6789 and MRN 99812" \
  --backend cactus \
  --policy hipaa-clinical
```

**Alternatively**, upload to Hugging Face Hub and use `cactus download`:

```bash
# Push LoRA + GGUF to Hub
huggingface-cli upload your-org/gemma4-pii-masker lora_adapter/
huggingface-cli upload your-org/gemma4-pii-masker gemma4-pii-unsloth.Q4_K_M.gguf

# Then on any machine with cactus:
cactus download your-org/gemma4-pii-masker --reconvert
```